<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Classification intuition

**[C05-M04-L02 · Classification intuition](https://learning.nextia-ai.com/courses/ml/m04/classification-intuition/)** · C05, Machine Learning: From Problem to Reliable Model · Module 4, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** see why a straight line cannot give a chance, turn ticket facts into a score from 0 to 1 with logistic regression, turn the score into a flag with a threshold, and see why the default threshold of 0.5 flags almost nothing.

| | |
|---|---|
| **Time** | About 60 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Linear regression and `fit`, from [Regression intuition](https://learning.nextia-ai.com/courses/ml/m04/regression-intuition/). Precision and recall, from [Classification metrics](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/). Scores, thresholds, AP and ROC AUC, from [The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/). The baselines, from [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). |
| **You do not need** | The local project. The setup below downloads the data here. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m04/classification-intuition/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M04-L02-classification-intuition/classification-intuition-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Why not a straight line?

Larkfield's question is yes or no: will this ticket be escalated within 72 hours? This is a **classification** task. The model should give each ticket a **score** from 0 to 1, like a weather app's chance of rain. A **threshold** then turns the score into a decision: if the score is at least the threshold, the ticket is **flagged** for the senior team.

Tomás suggests a shortcut: fit a linear regression to the 0/1 target, and call its prediction the chance. Try it with one feature, `word_count`.

> **Predict.** What will a straight line predict for a message of 2,000 words? Can it go above 1? Then run the cell.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

TARGET = "escalated_72h"
train = pd.read_csv("data/train.csv")
line = LinearRegression().fit(train[["word_count"]], train[TARGET])
print(f"line: {line.intercept_:.4f} + {line.coef_[0]:.5f} x word_count")
words = pd.DataFrame({"word_count": [0, 100, 200, 500, 1000, 2000]})
words["line"] = line.predict(words[["word_count"]]).round(3)
print(words.to_string(index=False))

> **Check.** You should see `line: 0.0488 + 0.00064 x word_count`, and `1.335` for 2,000 words.

Each word adds the same 0.00064, without a limit. A message of 2,000 words gets a "chance" of 134%. Now fit a line with four number features, and count the train tickets that get a value below 0. Run the cell.

In [ ]:
NUMBERS4 = ["priority", "word_count", "prior_tickets_90d", "created_hour"]
line4 = LinearRegression().fit(train[NUMBERS4], train[TARGET])
predicted4 = line4.predict(train[NUMBERS4])
print("train tickets below 0:", (predicted4 < 0).sum())
print("lowest value:", round(predicted4.min(), 3))

> **Check.** You should see `train tickets below 0: 570` and `lowest value: -0.038`.

A chance cannot be below 0 or above 1. A straight line has no limits, so it is the wrong shape for a chance. The lesson page shows it in a chart: [Why not a straight line?](https://learning.nextia-ai.com/courses/ml/m04/classification-intuition/#why-not-a-straight-line)

## 2. The S-curve: from a weighted sum to a score

**Logistic regression** makes the score in two steps:

1. A weighted sum, as in linear regression: `z = intercept + weight × feature`. `z` can be any number.
2. The **logistic function**, an S-curve, bends `z` into the range 0 to 1: `score = 1 / (1 + e^(−z))`.

> **Predict.** What score does `z = 0` give? And a very negative `z`, such as −4? Then run the cell.

In [ ]:
import math


def logistic(z):
    """Bend any number into the range 0 to 1."""
    return 1 / (1 + math.exp(-z))


for z in [-4, -2, -1, 0, 1, 2, 4]:
    print(f"z = {z:2d}  ->  score {logistic(z):.3f}")

> **Check.** You should see 0.018 for z = −4, 0.500 for z = 0 and 0.982 for z = 4.

`z = 0` always gives 0.5. A negative `z` gives a score below 0.5, and a positive `z` a score above it. The score never reaches 0 or 1. It is called logistic **regression** because inside it fits a weighted sum, like linear regression; "logistic" is the name of the S-curve.

## 3. One feature: priority

Load the validation set: the data you keep to check models while you work. Run the cell.

In [ ]:
valid = pd.read_csv("data/valid.csv")
print(len(train), "train tickets, escalation rate", round(train[TARGET].mean(), 3))
print(len(valid), "valid tickets, escalation rate", round(valid[TARGET].mean(), 3))

> **Check.** You should see `16939 train tickets, escalation rate 0.117` and `1179 valid tickets, escalation rate 0.155`.

Now fit a logistic regression with one feature, `priority` (1 is the most urgent, 3 the least). An **estimator** is a scikit-learn object that learns from data: `LinearRegression` and `LogisticRegression` are estimators. **fit** learns the numbers from the train set. Run the cell.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(train[["priority"]], train[TARGET])
print("intercept:", model.intercept_.round(4))
print("weight of priority:", model.coef_.round(4))

> **Check.** You should see `intercept: [-0.1249]` and `weight of priority: [[-0.8644]]`.

The weight is negative: a higher priority number (less urgent) gives a lower `z`, so a lower score.

> **Predict.** Compute the score of a priority-1 ticket by hand: z = −0.1249 + (−0.8644 × 1). Then run the cell.

In [ ]:
for priority in [1, 2, 3]:
    z = model.intercept_[0] + model.coef_[0][0] * priority
    rate = train.loc[train["priority"] == priority, TARGET].mean()
    print(f"priority {priority}: z = {z:.3f}, score {logistic(z):.3f}, train rate {rate:.3f}")

> **Check.** You should see `priority 1: z = -0.989, score 0.271, train rate 0.291`, then 0.135 for priority 2 and 0.062 for priority 3.

The scores are close to the escalation rates in train, but not equal: the model draws one straight line for `z` through all three priorities.

## 4. predict_proba and predict

An estimator has two ways to give an answer:

- **`predict_proba`** gives the scores. Column 1 (`[:, 1]`) is the score for "escalated".
- **`predict`** gives a decision, 0 or 1. It flags a ticket when its score is above 0.5.

> **Predict.** The highest score of this model is 0.271. How many valid tickets will `predict` flag? Then run the cell.

In [ ]:
scores = model.predict_proba(valid[["priority"]])[:, 1]
flags = model.predict(valid[["priority"]])
print("highest score:", round(scores.max(), 3))
print("flagged by predict:", flags.sum())
print("accuracy:", round((flags == valid[TARGET]).mean(), 3))

> **Check.** You should see `highest score: 0.271`, `flagged by predict: 0` and `accuracy: 0.845`.

No score reaches 0.5, so `predict` flags nothing. This is the majority baseline of [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/): 84.5% accuracy, and it catches no escalation. The scores still rank the tickets. Only the threshold is wrong.

> **Your turn.** Use a threshold of 0.2 instead. Count the flagged valid tickets (`scores >= 0.2`) and how many of them were escalated. Put the two numbers in `flagged_02` and `caught_02`. Run the cell, then the check cell.

In [ ]:
flags_02 = scores >= 0.2
flagged_02 = ...
caught_02 = ...
print(flagged_02, caught_02)

In [ ]:
expect_hash('[flagged_02, caught_02]', [flagged_02, caught_02], '334090d71d5c0f88')

The model flags the same 163 tickets as the rule "flag priority 1", and catches the same 61 escalations. With one feature, logistic regression is the rule with a number attached. More features can do better.

## 5. Several features

Use the four number features of section 1 again: `priority`, `word_count`, `prior_tickets_90d` and `created_hour`. They have a value for every ticket. Fit a logistic regression with all four. Run the cell.

In [ ]:
model4 = LogisticRegression().fit(train[NUMBERS4], train[TARGET])
print("intercept:", model4.intercept_.round(4))
print(pd.Series(model4.coef_[0], index=NUMBERS4).round(4))

> **Check.** You should see the intercept `[-0.7279]` and the weights −0.8938 (priority), 0.0052 (word_count), 0.0233 (prior_tickets_90d) and −0.0096 (created_hour).

The weights are in different units: 0.0052 per word, 0.0233 per earlier ticket. A small weight does not mean a small effect. A ticket with 300 words adds 300 × 0.0052 = 1.56 to `z`. [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/) puts all numbers on one scale.

Two metrics of [The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/) summarise how well the scores rank the tickets, for every threshold at once:

- **ROC AUC:** the chance that a random escalated ticket gets a higher score than a random other ticket. 0.5 is guessing.
- **Average precision (AP):** higher when the escalated tickets are near the top of the ranking. A constant score gets the escalation rate: 0.155 on valid.

Run the cell. It compares the one-feature model and the four-feature model.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

scores4 = model4.predict_proba(valid[NUMBERS4])[:, 1]
for name, s in [("priority only", scores), ("four numbers", scores4)]:
    auc = roc_auc_score(valid[TARGET], s)
    ap = average_precision_score(valid[TARGET], s)
    print(f"{name}: ROC AUC {auc:.3f}, AP {ap:.3f}, flagged at 0.5: {(s > 0.5).sum()}")

> **Check.** You should see `priority only: ROC AUC 0.658, AP 0.251, flagged at 0.5: 0` and `four numbers: ROC AUC 0.703, AP 0.312, flagged at 0.5: 6`.

The four-feature model ranks better. But at 0.5 it still flags only 6 of 1,179 tickets.

Why only four features? The other six are text categories (`channel`, `team`, `segment`, `region`) or numbers with gaps (`order_value`, `customer_tenure_days`). A logistic regression needs numbers with no gaps. Lesson 3, [Preprocess with a pipeline]({lesson_url('m04/preprocess-with-a-pipeline')}), shows each problem and solves them.

## 6. Change one thing: word_count instead of priority

> **Predict.** Will a model with only `word_count` rank better or worse than the priority model (ROC AUC 0.658)? Will it flag anything at 0.5?

> **Your turn.** Fit a logistic regression with only `word_count`. Put its valid ROC AUC, rounded to 3 decimals, in `auc_words`, and its count of flags from `predict` in `flagged_words`. Run the cell, then the check cell.

In [ ]:
model_words = LogisticRegression()
...  # fit model_words on train[["word_count"]]
auc_words = ...
flagged_words = ...
print(auc_words, flagged_words)

In [ ]:
expect_hash('[auc_words, flagged_words]', [auc_words, flagged_words], '140b371942a82a23')

`word_count` alone ranks worse than priority: ROC AUC 0.598. It flags 1 ticket at 0.5, a ticket with a very long message, and that ticket was not escalated. The S-curve keeps the score below 1. But the weight (+0.0049 per word) still adds up with every word, so a few extreme tickets get a high score.

## 7. Failure case: predict as the decision

Tomás, the AI engineer, reports the four-feature model with `predict`: "accuracy 84.3%". Run the cell.

In [ ]:
flags4 = model4.predict(valid[NUMBERS4])
caught = int((flags4 & (valid[TARGET] == 1)).sum())
print("flagged:", flags4.sum(), "of", len(valid))
print("caught:", caught, "of", valid[TARGET].sum(), "escalations")
print("accuracy:", round((flags4 == valid[TARGET]).mean(), 3))

> **Check.** You should see `flagged: 6 of 1179`, `caught: 2 of 183 escalations` and `accuracy: 0.843`.

The accuracy is lower than "never escalate" (0.845), and the model catches 2 escalations of 183. **Cause:** `predict` uses 0.5. With 15.5% escalations, very few tickets reach 0.5. **Fix:** use `predict_proba` and choose the threshold from Grace's capacity (at most 20% of tickets). [Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/) does this. For now, at 0.2 this model flags 192 tickets (16.3%) and catches 73.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m04/classification-intuition/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you saw that a straight line cannot give a chance (1.335 for 2,000 words; 570 train tickets below 0). You made a score from a weighted sum and the logistic function, and checked one by hand (0.271 for priority 1). `predict_proba` gives scores; `predict` uses a threshold of 0.5. On Larkfield's data, 0.5 flags almost nothing. It flags 0 tickets with priority alone, and 6 with four features. Four features rank better than one (ROC AUC 0.703 against 0.658). Text columns and gaps need preprocessing, which is the next lesson.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m04/classification-intuition/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.